In [ ]:
print("hello")

In [ ]:
import torch
if torch.cuda.is_available():
    print("Using GPU")
else:
    print("Using CPU")
import numpy as np
import matplotlib.pyplot as plt

groundData = torch.randn(1000,1024)
singleQueryData = torch.randn(800,1024)

nus = [0.3]
row = 0
index = 1

for nu in nus:
    print("C++ ------------------------------")
    obj = FacilityLocationVariantMutualInformationFunction(n=groundData.shape[0], 
                                            num_queries=singleQueryData.shape[0],
                                            data=groundData,
                                            queryData=singleQueryData,
                                            metric="cosine",
                                            
                                            queryDiversityEta=nu,)
    greedyList = obj.maximize(budget=10,optimizer='NaiveGreedy', stopIfZeroGain=False, 
                                stopIfNegativeGain=False, verbose=False)
    print(greedyList)
        
    print("Pytorch --------------------------------")

    obj2 = FacilityLocationVariantMutualInformation(n=groundData.shape[0], 
                                            num_queries=singleQueryData.shape[0], 
                                            data=groundData,
                                            queryData=singleQueryData,
                                            metric="cosine", 
                                            queryDiversityEta=nu)
    greedyList2 = obj2.maximize(budget=10,optimizer='NaiveGreedy', stopIfZeroGain=False, 
                                stopIfNegativeGain=False, verbose=False)
    print(greedyList2)
    idxs_cpp = torch.tensor([p[0] for p in greedyList], dtype=torch.long)
    idxs_py  = torch.tensor([p[0] for p in greedyList2], dtype=torch.long)

    gains_cpp = torch.tensor([float(p[1]) for p in greedyList], dtype=torch.float16)
    gains_py  = torch.tensor([float(p[1]) for p in greedyList2], dtype=torch.float16)

    same_indices = torch.equal(idxs_cpp, idxs_py)
    same_gains = torch.allclose(gains_cpp, gains_py, rtol=1e-5, atol=1e-7)

    print("indices match:", same_indices)
    print("gains match:", same_gains)